# SpecCounter

Used to train and evaluate SpecCounter, as well as TSSMCounter pretrained variant.

In [ ]:
#Imports and config

import os
import csv
import json
import tqdm
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchaudio
import torchaudio.functional as FTF

from preprocessing.data_converter import DataConverter

np.random.seed(0)
torch.manual_seed(0)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


OUTPUT_DIR = "/users/hani/AudioCounting/models" #where to save models and embeddings
RESULTS_FILE = "/users/hani/AudioCounting/results_new/speccounter_results.txt"

DINO_MODEL = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEVICE).eval()
WAV2VEC_MODEL = torchaudio.pipelines.WAV2VEC2_BASE.get_model().to(DEVICE).eval()
DATA_CONVERTER = DataConverter()

In [2]:
#Transforms and data representations

def get_transforms(rep):
    #general transform selector for spectrograms and TSSMs
    if rep == "spec":
        return T.Compose([
            T.Lambda(lambda img: TF.pad(
                img,
                padding=(
                    0,
                    (14 - img.shape[-2] % 14) % 14,
                    (14 - img.shape[-1] % 14) % 14,
                    0,
                )
            )),
            T.Lambda(lambda img: img.repeat(3, 1, 1) if img.shape[0] == 1 else img)
        ])
    elif rep == "tssm":
        return T.Compose([
            T.Resize((504, 504), interpolation=T.InterpolationMode.BICUBIC),
            T.Lambda(lambda img: img.repeat(3, 1, 1) if img.shape[0] == 1 else img)
        ])

def create_spectrogram(audio, sr):
    y = audio.cpu().numpy()
    if y.ndim > 1:
        y = y.mean(axis=0)
    
    _, _, S = signal.spectrogram(y, sr, nperseg=512, noverlap=256)
    S = np.log(S + 1e-7)
    S = DATA_CONVERTER.apply_histogram_equalisation(S, method="global")
    
    mean, std = np.mean(S), np.std(S)
    S = (S - mean) / (std + 1e-9)
    return torch.tensor(S, dtype=torch.float32).unsqueeze(0)

def create_tssm(audio):
    with torch.no_grad():
        features, _ = WAV2VEC_MODEL(audio.unsqueeze(0))
        x = features[0]
        x = F.normalize(x, p=2, dim=-1)
        tssm = (x @ x.T).cpu().numpy()
        
    tssm = DATA_CONVERTER.apply_histogram_equalisation(tssm)
    return torch.tensor(tssm, dtype=torch.float32).unsqueeze(0)

    

In [ ]:
#DINOv2 Embeddings functions and dataset

class DINOv2Dataset(torch.utils.data.Dataset):
    #creates embeddings for spectrograms or TSSMs using DINOv2
    def __init__(self, root_dir, rep, is_wav=True, csv_path=None, filter_reps=True):
        self.root_dir = root_dir
        self.rep = rep
        self.is_wav = is_wav
        self.cut_start = "clocks" in root_dir
        self.volume_shift = "dolphins" in root_dir
        self.transform = get_transforms(rep)
        
        self.metadata = {}
        if csv_path:
            with open(csv_path, 'r') as f:
                reader = csv.DictReader(f)
                if filter_reps:
                    self.metadata = {row["location"]: row for row in reader if 0 <= int(row.get("repetitions", 0)) <= 8}
                else:
                    self.metadata = {row["location"]: row for row in reader}

        ext = ".wav" if is_wav else ".npy"
        all_files = [f for f in os.listdir(root_dir) if f.endswith(ext) and os.path.getsize(os.path.join(root_dir, f)) > 0]
        
        if self.metadata:
            self.files = sorted([f for f in all_files if f in self.metadata])
        else:
            self.files = sorted(all_files)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]
        file_path = os.path.join(self.root_dir, file_name)
        
        if self.is_wav:
            audio, sr = torchaudio.load(file_path)
            if sr != 16000:
                audio = FTF.resample(audio, sr, 16000)
                sr = 16000
                
            audio = torch.mean(audio, dim=0) if audio.shape[0] > 1 else audio.squeeze(0)
            
            if self.cut_start and self.metadata:
                start_time = float(self.metadata[file_name].get('start_time', 0))
                audio = audio[int(start_time * sr):]
                
            audio = audio.to(DEVICE)
            if self.volume_shift:
                audio /= audio.abs().max() + 1e-7

            if self.rep == "spec":
                feature_tensor = create_spectrogram(audio, sr)
            elif self.rep == "tssm":
                feature_tensor = create_tssm(audio)
        else:
            arr = np.load(file_path)
            if arr.ndim == 2:
                arr = np.expand_dims(arr, axis=0)
            arr = arr.astype(np.float32)
            arr = arr / (arr.max() + 1e-8)
            feature_tensor = torch.tensor(arr, dtype=torch.float32)

        input_tensor = self.transform(feature_tensor).unsqueeze(0).to(DEVICE)
        
        with torch.inference_mode():
            embedding = DINO_MODEL(input_tensor).squeeze(0).cpu().numpy()

        return embedding, file_name

def compute_embeddings(dataset, save_name, split):
    #create DINOv2 embeddings and save as .npy
    output_path = os.path.join(OUTPUT_DIR, save_name, f"dinov2_embeddings_{split}.npy")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    embeddings_list = []
    
    print(f"Computing embeddings for {save_name}, split: {split}...")
    for embedding, _ in tqdm.tqdm(dataset):
        embeddings_list.append(embedding)

    np.save(output_path, np.array(embeddings_list))
    print(f"Saved to {output_path}")

In [33]:
#Dataset for loading DINOv2 embeddings once computed

class Dinov2EmbeddingDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split, embedding_name, csv_path=None, use_json=False):
        
        self.data_dir = os.path.join(root_dir, split)
        if not os.path.exists(self.data_dir):
            self.data_dir = root_dir
            
        self.embeddings = np.load(os.path.join(OUTPUT_DIR, embedding_name, f"dinov2_embeddings_{split}.npy"))
        
        ext = ".npy" if "spec" in root_dir or "tssm" in root_dir else ".wav"
        self.files = sorted([f for f in os.listdir(self.data_dir) if f.endswith(ext) and os.path.getsize(os.path.join(self.data_dir, f)) > 0])
        
        self.labels = []
        keep_file_indices = []
        keep_embed_indices = []
        embed_idx = 0

        metadata = {}
        if use_json:
            json_path = os.path.join(self.data_dir.replace("tssm", "wav").replace("spec", "wav"), 'metadata.json')
            with open(json_path, 'r') as f:
                metadata = json.load(f)
        elif csv_path:
            with open(csv_path, 'r') as f:
                reader = csv.DictReader(f)
                metadata = {row["location"]: int(row["repetitions"]) for row in reader}

        for i, fname in enumerate(self.files):
            if use_json:
                label = metadata[fname.replace(ext, "")]['num_repetitions']
            elif csv_path:
                if fname not in metadata: continue
                label = metadata[fname]
            # else:
            #     label = int(fname.split("_")[-1].split(".")[0])
            
            if 0 <= label <= 8:
                keep_file_indices.append(i)
                keep_embed_indices.append(embed_idx)
                self.labels.append(label)

            embed_idx += 1

        self.embeddings = self.embeddings[keep_embed_indices, :]
        self.files = [self.files[i] for i in keep_file_indices]
        
    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        return torch.tensor(self.embeddings[idx, :], dtype=torch.float32), self.labels[idx]

In [5]:
class MLPHead(nn.Module):
    def __init__(self, in_features=384, out_features=9):
        super(MLPHead, self).__init__()
        self.layer1 = nn.Linear(in_features, 256, bias=True)
        self.layer2 = nn.Linear(256, 128, bias=True)
        self.layer3 = nn.Linear(128, 128, bias=True)
        self.layer4 = nn.Linear(128, out_features, bias=True)

    def forward(self, x):
        x = F.leaky_relu(self.layer1(x))
        x = F.leaky_relu(self.layer2(x))
        x = F.leaky_relu(self.layer3(x))
        return F.log_softmax(self.layer4(x), dim=-1)

In [ ]:
#train and test functions

def train_model(model_name, train_loader, val_loader, epochs=30, lr=0.001):
    model = MLPHead().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_obo_acc = -1.0
    
    print(f"Training {model_name} -------- \n")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target, reduction='sum')
            train_loss += loss.item()
            loss.backward()
            optimizer.step()
            
        #val
        model.eval()
        correct, obo_correct = 0, 0
        with torch.inference_mode():
            for data, target in val_loader:
                data, target = data.to(DEVICE), target.to(DEVICE)
                output = model(data)
                pred = output.max(1, keepdim=True)[1]
                correct += pred.eq(target.view_as(pred)).sum().item()
                obo_correct += (torch.abs(pred.flatten() - target.flatten()) <= 1).sum().item()
        
        obo_acc = 100.0 * obo_correct / len(val_loader.dataset)
        if obo_acc > best_obo_acc:
            best_obo_acc = obo_acc
            os.makedirs(os.path.join(OUTPUT_DIR, model_name), exist_ok=True)
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, model_name, f"model_{model_name}.pth"))
            print(f"Epoch {epoch} | New Best OBO: {best_obo_acc:.2f}%")
        else:
            print(f"Epoch {epoch} | OBO: {obo_acc:.2f}% (Best: {best_obo_acc:.2f}%)")

def test_model(model_name, test_loader, test_set_name):
    model = MLPHead().to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, model_name, f"model_{model_name}.pth"), map_location=DEVICE))
    model.eval()

    test_loss, correct, obo_correct, mae_sum = 0, 0, 0, 0.0
    with torch.inference_mode():
        for data, target in tqdm.tqdm(test_loader, desc=f"Evaluating {test_set_name}"):
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()
            obo_correct += (torch.abs(pred.flatten() - target.flatten()) <= 1).sum().item()
            
            denom = target.view_as(pred).float() + 0.1
            mae_sum += (torch.abs(pred - target.view_as(pred)).float() / denom).sum().item()

    n_samples = len(test_loader.dataset)
    acc = 100.0 * correct / n_samples
    obo_acc = 100.0 * obo_correct / n_samples
    mae = mae_sum / n_samples

    result_str = f"Model: {model_name} | Dataset: {test_set_name} | MAE: {mae:.4f} | Acc: {acc:.2f}% | OBO: {obo_acc:.2f}% |"
    print(result_str)
    
    with open(RESULTS_FILE, "a") as f:
        f.write(result_str + "\n")

def test_one_sample(model_name, wav_path, rep):
    model = MLPHead().to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, model_name, f"model_{model_name}.pth"), map_location=DEVICE))
    model.eval()

    audio, sr = torchaudio.load(wav_path)
    if sr != 16000:
        audio = FTF.resample(audio, sr, 16000)
        sr = 16000
    audio = torch.mean(audio, dim=0) if audio.shape[0] > 1 else audio.squeeze(0)
    audio = audio.to(DEVICE)
    if rep == "spec":
        feature_tensor = create_spectrogram(audio, sr)
    elif rep == "tssm":
        feature_tensor = create_tssm(audio)

    input_tensor = get_transforms(rep)(feature_tensor).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        embedding = DINO_MODEL(input_tensor)
        output = model(embedding)
        pred = output.max(1, keepdim=True)[1].item()
    print(f"Predicted count: {pred}")

    plt.figure(figsize=(10, 4))
    plt.title("Waveform")
    plt.plot(audio.cpu().numpy())
    plt.show()

    if rep == "spec":
        plt.figure(figsize=(10, 4))
    else:
        plt.figure(figsize=(6, 6))
    plt.title("Representation")
    plt.imshow(feature_tensor.squeeze(0).cpu().numpy(), aspect='auto', origin='upper', cmap="magma")
    plt.tight_layout()
    plt.show()


In [ ]:
#create embeddings for synthetic datasets
dataset_name = "/scratch/local/ssd/hani/RS" #dataset name where spec/tssm folders are located
rep = "spec"
embedding_name = "SpecCounter-RS" #where to save embeddings

for split in ["train", "val", "test"]:
    split_dir = os.path.join(dataset_name, rep, split)
    
    dataset = DINOv2Dataset(split_dir, rep, is_wav=False)
    
    compute_embeddings(dataset, save_name=embedding_name, split=split)

In [ ]:
#create embeddings for real datasets

# uncomment rep and dataset info to create embeddings for

#-------------------------------------------------------------------

# rep = "spec"

# embedding_name = "heartbeats_spec"
# dataset_name = "/scratch/local/hdd/hani/heartbeats/wav/"
# csv_path = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv"

# embedding_name = "dolphins_spec"
# dataset_name = "/scratch/local/hdd/hani/dolphins/test_padded/"
# csv_path = "/users/hani/AudioCounting/preprocessing/dolphins/dolphins.csv"

# embedding_name = "clocks_spec"
# dataset_name = "/scratch/local/hdd/hani/bbc_clocks/audio"
# csv_path = "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"

#-------------------------------------------------------------------

rep = "tssm"

# embedding_name = "heartbeats_tssm"
# dataset_name = "/scratch/local/hdd/hani/heartbeats/wav/"
# csv_path = "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv"

# embedding_name = "dolphins_tssm"
# dataset_name = "/scratch/local/hdd/hani/dolphins/test_padded/"
# csv_path = "/users/hani/AudioCounting/preprocessing/dolphins/dolphins.csv"

embedding_name = "clocks_tssm"
dataset_name = "/scratch/local/hdd/hani/bbc_clocks/audio"
csv_path = "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"

#-------------------------------------------------------------------

dataset = DINOv2Dataset(dataset_name, rep, is_wav=True, csv_path=csv_path, filter_reps=False)

compute_embeddings(dataset, save_name=embedding_name, split="test")

In [ ]:
#train classifier
#spec is SpecCounter model, tssm is TSSMCounter variant with pretrained dinov2
model_name = "RS-A"

embedding_name = "RS-A" #where to save model weights
dataset_name = "/scratch/local/ssd/hani/RS" #which dataset to train on
rep = "spec" #or tssm

batch_size = 64
epochs = 30
lr = 0.001

dataset_root = os.path.join(dataset_name, rep)

print(f"Loading precomputed embeddings from directory: {dataset_root}")

train_dataset = Dinov2EmbeddingDataset(dataset_root, "train", embedding_name, use_json=True)
val_dataset = Dinov2EmbeddingDataset(dataset_root, "val", embedding_name, use_json=True)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False)

train_model(model_name, train_loader, val_loader, epochs=epochs, lr=lr)

In [ ]:
#choose model and test
model_name = "RS-A"
rep = "spec"

# dataset_name = "/scratch/local/ssd/hani/RS" #where original spec/tssm folders are located
# embedding_name = "RS-A" #where embeddings are stored in OUTPUT_DIR

# embedding_name = "heartbeats_spec"
# dataset_name = "/scratch/local/hdd/hani/heartbeats/wav/"

embedding_name = "dolphins_spec"
dataset_name = "/scratch/local/hdd/hani/dolphins/test_padded/"

# embedding_name = "clocks_spec"
# dataset_name = "/scratch/local/hdd/hani/bbc_clocks/audio"

In [ ]:
#test model

csv_dict = {
    "dolphins": "/users/hani/AudioCounting/preprocessing/dolphins/dolphins.csv",
    "heartbeats": "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv",
    "clocks": "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"
}


# instantiate dataset class 
real_words = ["dolphins", "heartbeats", "clocks"]
real_set = next((word for word in real_words if word in dataset_name), None)

if real_set:
    test_dataset = Dinov2EmbeddingDataset(root_dir=dataset_name, split="test", embedding_name=embedding_name, csv_path=csv_dict[real_set])

else:
    # assume synthetic test set (RS/RSN/RVN)
    test_dataset = Dinov2EmbeddingDataset(root_dir=os.path.join(dataset_name, rep), split="test", embedding_name=embedding_name, use_json=True)


test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
test_model(model_name=model_name, test_loader=test_loader, test_set_name=dataset_name)

In [ ]:
#test one sample
wav_path = "/scratch/local/ssd/hani/RS/wav/test/009200.wav"
model_name = "RS-A"
rep = "spec"

test_one_sample(model_name=model_name, wav_path=wav_path, rep=rep)

In [ ]:
#test SpecCounter on all datasets

models_to_test = ["RS-A", "RSN-A", "RVN-A"]

datasets_to_test = {
    "/scratch/local/ssd/hani/RS": "RS-A",
    "/scratch/local/ssd/hani/RSN": "RSN-A",
    "/scratch/local/ssd/hani/RVN": "RVN-A",
    "/scratch/local/hdd/hani/heartbeats/wav/": "heartbeats_spec",
    "/scratch/local/hdd/hani/dolphins/test_padded/": "dolphins_spec",
    "/scratch/local/hdd/hani/bbc_clocks/audio": "clocks_spec"
}

csv_dict = {
    "dolphins": "/users/hani/AudioCounting/preprocessing/dolphins/dolphins.csv",
    "heartbeats": "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv",
    "clocks": "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"
}


real_words = ["dolphins", "heartbeats", "clocks"]

for model_name in models_to_test:
    for dataset_name, embedding_name in datasets_to_test.items():
        real_set = next((word for word in real_words if word in dataset_name), None)
        
        if real_set:
            test_dataset = Dinov2EmbeddingDataset(root_dir=dataset_name, split="test", embedding_name=embedding_name, csv_path=csv_dict[real_set])
        else:
            test_dataset = Dinov2EmbeddingDataset(root_dir=os.path.join(dataset_name, rep), split="test", embedding_name=embedding_name, use_json=True)

        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
        test_model(model_name=model_name, test_loader=test_loader, test_set_name=dataset_name)


In [ ]:
#test TSSMCounter variant on all datasets

models_to_test = ["RS-B", "RSN-B", "RVN-B"]

datasets_to_test = {
    "/scratch/local/ssd/hani/RS": "RS-B",
    "/scratch/local/ssd/hani/RSN": "RSN-B",
    "/scratch/local/ssd/hani/RVN": "RVN-B",
    "/scratch/local/hdd/hani/heartbeats/wav/": "heartbeats_tssm",
    "/scratch/local/hdd/hani/dolphins/test_padded/": "dolphins_tssm",
    "/scratch/local/hdd/hani/bbc_clocks/audio": "clocks_tssm"
}

csv_dict = {
    "dolphins": "/users/hani/AudioCounting/preprocessing/dolphins/dolphins.csv",
    "heartbeats": "/users/hani/AudioCounting/preprocessing/heartbeats/heartbeats_sorted.csv",
    "clocks": "/users/hani/AudioCounting/preprocessing/bbc_clocks/bbc_clocks.csv"
}

real_words = ["dolphins", "heartbeats", "clocks"]

for model_name in models_to_test:
    for dataset_name, embedding_name in datasets_to_test.items():
        real_set = next((word for word in real_words if word in dataset_name), None)
        
        if real_set:
            test_dataset = Dinov2EmbeddingDataset(root_dir=dataset_name, split="test", embedding_name=embedding_name, csv_path=csv_dict[real_set])
        else:
            test_dataset = Dinov2EmbeddingDataset(root_dir=os.path.join(dataset_name, rep), split="test", embedding_name=embedding_name, use_json=True)

        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
        test_model(model_name=model_name, test_loader=test_loader, test_set_name=dataset_name)